# 3. Resolved images from an `artefact` model

`RTModel.make_image` turns the radial model into a sky image at any wavelength inside the model grid. Thermal emission is isotropic; scattered light is weighted by a Henyey-Greenstein phase function. This notebook covers:

1. making a model to image (an HD 104860-like belt),
2. thermal and scattered-light images,
3. inclination, position angle and disc thickness,
4. beam convolution and units,
5. writing a FITS file.

`make_image` needs the emission and/or scattering to have been calculated, so run the SED workflow first.

In [ ]:
import sys
from pathlib import Path

# The notebooks live in notebooks/, the helpers in example/
sys.path.insert(0, str(Path.cwd().parent / "example"))

import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline

In [ ]:
from example_utils import OUTPUT, base_parameters, run_model, show_image

In [ ]:
model = run_model(base_parameters(
    lstar=1.318, rstar=0.94, tstar=5950.0, dstar=48.0,
    amin=4.3, mdust=7.2e-3,
    dtype="onepl", rin=60.0, rout=170.0, alpha_out=0.0, nring=41,
))

## Thermal and scattered light

Images are north up, east left. The default field of view is 2.4 outer radii. `beam` is a Gaussian FWHM in arcsec (or a PSF array on the image grid).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
show_image(model.make_image(70.0, component="emit", inc=60, pa=30, beam=5.0),
           ax=axes[0], title="70 um thermal, 5\" beam")
show_image(model.make_image(1300.0, component="emit", inc=60, pa=30, beam=1.0),
           ax=axes[1], title="1.3 mm thermal, 1\" beam")
show_image(model.make_image(1.6, component="scat", inc=60, pa=30),
           ax=axes[2], log=True, title="1.6 um scattered light")
plt.show()

## Inclination and position angle

`inc` is 0 for face-on up to (but excluding) 90 degrees edge-on. `pa` is the position angle of the major axis, east of north.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4.2))
for ax, (inc, pa) in zip(axes, [(0, 0), (45, 0), (45, 90), (80, 30)]):
    show_image(model.make_image(70.0, component="emit", inc=inc, pa=pa),
               ax=ax, title=f"inc={inc}, PA={pa}")
plt.tight_layout()
plt.show()

## Disc thickness and scattering phase function

The vertical profile is Gaussian with scale height `aspect * r`. It matters most for nearly edge-on discs, where it sets the apparent thickness. The scattered-light phase function is a two-component Henyey-Greenstein with asymmetry `g1` (forward), `g2` (backward) and weight `hg_weight`; the defaults are placeholders to be set from observations. Setting `g1 = g2 = 0` gives isotropic scattering.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
show_image(model.make_image(70.0, component="emit", inc=85, pa=0, aspect=0.02),
           ax=axes[0], title="aspect = 0.02")
show_image(model.make_image(70.0, component="emit", inc=85, pa=0, aspect=0.1),
           ax=axes[1], title="aspect = 0.10")
show_image(model.make_image(1.6, component="scat", inc=60, pa=0, g1=0.0, g2=0.0),
           ax=axes[2], log=True, title="isotropic scattering")
plt.show()

## Units and beams

`unit` can be `"mJy/pixel"` (default), `"mJy/arcsec2"` or `"mJy/beam"` (needs a `beam`). Convolution preserves total flux in mJy/pixel.

In [ ]:
image = model.make_image(850.0, inc=60, pa=30, beam=2.0, unit="mJy/beam")
print(f"pixel scale {image.pixscale:.3f} arcsec, peak {image.data.max():.3g} {image.unit}")

fig, ax = plt.subplots(figsize=(5, 5))
show_image(image, ax=ax)
plt.show()

## Radial profile along the major axis

In [ ]:
sharp = model.make_image(850.0, component="emit", inc=60, pa=90, unit="mJy/arcsec2")
row = sharp.data.shape[0] // 2
offset = (np.arange(sharp.data.shape[1]) - sharp.data.shape[1] / 2 + 0.5) * sharp.pixscale

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(offset, sharp.data[row])
ax.set_xlabel("Offset along the major axis (arcsec)")
ax.set_ylabel(sharp.unit)
plt.show()

## FITS output

`ModelImage.writeto` writes a FITS file with a tangent-plane WCS, the beam (if circular) and the model geometry in the header.

In [ ]:
OUTPUT.mkdir(parents=True, exist_ok=True)
path = OUTPUT / "03_hd104860_850um.fits"
image.writeto(path)
print("Wrote", path)